# 81: STH-MVRV Z-Score Exits (Regime-Adaptive)

**Problem from Notebook 80:** Fixed STH-MVRV thresholds don't work:
- STH-MVRV > 1.9 never triggered in our backtest period
- STH-MVRV > 1.7 caused -267% underperformance
- STH-MVRV > 1.5 hurt even more (-458%)

## The Solution: Z-Score Normalization

**Z-Score = (STH-MVRV - Mean) / StdDev**

This adapts to market regimes:
- What's "hot" in 2017 (STH-MVRV 1.6) might be normal in 2024 (STH-MVRV 1.8)
- Z-score tells you how extreme *relative to recent history*
- Captures outliers that signal local tops

## Z-Score Interpretation:

| Z-Score | Percentile | Meaning | Action |
|---------|------------|---------|--------|
| **< -1.5** | 6.7th | Deep cooled, extreme capitulation | **STRONG BUY** |
| **-1.5 to -1.0** | 6.7-15.9th | Cooled, accumulation zone | **BUY** |
| **-1.0 to +1.0** | 15.9-84.1th | Fair value, normal range | **HOLD** |
| **+1.0 to +1.5** | 84.1-93.3rd | Warming, elevated euphoria | **CAUTION** |
| **+1.5 to +2.0** | 93.3-97.7th | Overheated, local top risk | **CONSIDER EXIT** |
| **> +2.0** | 97.7th+ | Extreme euphoria, danger zone | **EXIT** |

## What We'll Test:

**Entry:** Buy The Dip (4/5 conditions) - unchanged

**Exit strategies:**
1. Never Exit (baseline)
2. STH-MVRV Z > 1.0 (84th percentile, early exit)
3. STH-MVRV Z > 1.5 (93rd percentile, local tops)
4. STH-MVRV Z > 2.0 (98th percentile, extreme)
5. STH-MVRV Z > 2.5 (99th percentile, very conservative)
6. Original (MVRV>2.0 AND LTH-SOPR>1.5)

**Rolling windows to test:**
- 365 days (1 year lookback)
- 730 days (2 year lookback)
- Full history (expanding window)

**Expected:** Z > 1.5-2.0 should capture regime-relative tops across all market cycles.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

print(f"✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
print(f"  STH-MVRV available: {df['mvrv_sth'].notna().sum()} days")
print(f"  STH-MVRV range: {df['mvrv_sth'].min():.3f} to {df['mvrv_sth'].max():.3f}")
df.head()

## 2. Calculate STH-MVRV Z-Scores (Multiple Windows)

In [ ]:
print("Calculating STH-MVRV Z-scores...\n")

# Calculate Z-scores with different lookback windows
sth_mvrv = df['mvrv_sth']

# 1. Rolling 365-day window (1 year)
sth_mean_365 = sth_mvrv.rolling(365, min_periods=30).mean()
sth_std_365 = sth_mvrv.rolling(365, min_periods=30).std()
sth_z_365 = (sth_mvrv - sth_mean_365) / sth_std_365

# 2. Rolling 730-day window (2 years)
sth_mean_730 = sth_mvrv.rolling(730, min_periods=90).mean()
sth_std_730 = sth_mvrv.rolling(730, min_periods=90).std()
sth_z_730 = (sth_mvrv - sth_mean_730) / sth_std_730

# 3. Expanding window (full history)
sth_mean_expanding = sth_mvrv.expanding(min_periods=90).mean()
sth_std_expanding = sth_mvrv.expanding(min_periods=90).std()
sth_z_expanding = (sth_mvrv - sth_mean_expanding) / sth_std_expanding

# Store in dataframe
df['sth_z_365'] = sth_z_365
df['sth_z_730'] = sth_z_730
df['sth_z_expanding'] = sth_z_expanding

# Statistics
print("Z-Score Statistics:")
print("="*70)
for name, z_score in [('365-day', sth_z_365), ('730-day', sth_z_730), ('Expanding', sth_z_expanding)]:
    z_clean = z_score.dropna()
    print(f"\n{name} window:")
    print(f"  Mean:   {z_clean.mean():.3f} (should be ~0)")
    print(f"  StdDev: {z_clean.std():.3f} (should be ~1)")
    print(f"  Min:    {z_clean.min():.3f}")
    print(f"  Max:    {z_clean.max():.3f}")
    print(f"  Percentiles:")
    print(f"    5th:  {z_clean.quantile(0.05):.3f}")
    print(f"    50th: {z_clean.quantile(0.50):.3f}")
    print(f"    95th: {z_clean.quantile(0.95):.3f}")
    print(f"    99th: {z_clean.quantile(0.99):.3f}")
    
    # Time in zones
    overheated = (z_clean > 2.0).sum()
    warming = ((z_clean > 1.5) & (z_clean <= 2.0)).sum()
    cooled = (z_clean < -1.5).sum()
    print(f"  Time Z > 2.0 (overheated): {overheated} days ({overheated/len(z_clean)*100:.1f}%)")
    print(f"  Time Z > 1.5 (warming): {warming} days ({warming/len(z_clean)*100:.1f}%)")
    print(f"  Time Z < -1.5 (cooled): {cooled} days ({cooled/len(z_clean)*100:.1f}%)")

## 3. Visualize Z-Scores Over Time

In [ ]:
# Plot Z-scores
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Price
ax1 = axes[0]
ax1.plot(df.index, df['price'], color='black', linewidth=2, label='BTC Price')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['sth_z_730'] > 2.0), alpha=0.3, color='red', label='Z > 2.0 (Overheated)')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['sth_z_730'] < -1.5), alpha=0.3, color='green', label='Z < -1.5 (Cooled)')
ax1.set_ylabel('BTC Price ($)', fontsize=12)
ax1.set_title('STH-MVRV Z-Score: Market Regimes', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# STH-MVRV absolute value
ax2 = axes[1]
ax2.plot(df.index, df['mvrv_sth'], color='blue', linewidth=1.5, label='STH-MVRV (Absolute)')
ax2.axhline(1.0, color='gray', linestyle='-', alpha=0.5, label='Equilibrium')
ax2.set_ylabel('STH-MVRV', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Z-scores (730-day window)
ax3 = axes[2]
ax3.plot(df.index, df['sth_z_730'], color='purple', linewidth=1.5, label='STH-MVRV Z-Score (730d)')
ax3.axhline(0, color='gray', linestyle='-', alpha=0.5, label='Mean')
ax3.axhline(1.5, color='orange', linestyle='--', alpha=0.7, label='Z = 1.5 (93rd %ile)')
ax3.axhline(2.0, color='red', linestyle='--', alpha=0.7, label='Z = 2.0 (98th %ile)')
ax3.axhline(-1.5, color='green', linestyle='--', alpha=0.7, label='Z = -1.5 (7th %ile)')
ax3.fill_between(df.index, -5, 5, where=(df['sth_z_730'] > 2.0), alpha=0.2, color='red')
ax3.fill_between(df.index, -5, 5, where=(df['sth_z_730'] < -1.5), alpha=0.2, color='green')
ax3.set_ylabel('Z-Score', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.set_ylim(-3, 4)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Green = Deep buying opportunity (Z < -1.5)")
print("📊 Orange = Warming (Z > 1.5)")
print("📊 Red = Overheated (Z > 2.0)")

## 4. Generate Entry Signals

In [ ]:
# Entry: Buy The Dip (4/5 conditions)
print("Generating entry signals...\n")

c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0
c5 = (df['liq_long'] / df['liq_short']) > 1.0

c4 = c4.fillna(False)
c5 = c5.fillna(False)

entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = (entry_count >= 4).fillna(False).astype(bool)

print(f"✓ Entry signals: {entries.sum()}")

## 5. Generate Exit Signals (Z-Score Based)

In [ ]:
print("Generating Z-score exit signals...\n")

# We'll focus on 730-day window (2 years) as primary
# But also test 365-day and expanding

exit_strategies = {}

# Never exit baseline
exit_strategies['never'] = (pd.Series(False, index=df.index, dtype=bool), 'Never Exit')

# 730-day window (primary test)
for threshold, name_suffix in [(1.0, '84th%'), (1.5, '93rd%'), (2.0, '98th%'), (2.5, '99th%')]:
    exits = (df['sth_z_730'] > threshold).fillna(False).astype(bool)
    exit_strategies[f'z730_{threshold}'] = (exits, f'Z730 > {threshold} ({name_suffix})')

# 365-day window (shorter, more reactive)
for threshold in [1.5, 2.0]:
    exits = (df['sth_z_365'] > threshold).fillna(False).astype(bool)
    exit_strategies[f'z365_{threshold}'] = (exits, f'Z365 > {threshold}')

# Expanding window (full history)
for threshold in [1.5, 2.0]:
    exits = (df['sth_z_expanding'] > threshold).fillna(False).astype(bool)
    exit_strategies[f'zexp_{threshold}'] = (exits, f'Z-Expanding > {threshold}')

# Original for comparison
exit_strategies['original'] = (((df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)).fillna(False).astype(bool), 'Original (MVRV>2.0)')

print("Exit signal counts:")
print("="*70)
for key, (exits, name) in exit_strategies.items():
    print(f"{name:<40} {exits.sum():>4} signals")

# Verify dtypes
print("\nDtype verification:")
for key, (exits, name) in exit_strategies.items():
    print(f"  {key}: {exits.dtype}")

## 6. Backtest All Strategies

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }

print("\n" + "="*90)
print("BACKTESTING: STH-MVRV Z-SCORE EXITS")
print("="*90)

# Backtest all strategies
results = {}
for key, (exits, name) in exit_strategies.items():
    results[key] = backtest_strategy(df, entries, exits, name)

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100

# Sort by return
sorted_results = sorted(results.items(), key=lambda x: x[1]['total_return'], reverse=True)

# Results table
print(f"\n{'Strategy':<40} {'Return':>12} {'Sharpe':>8} {'Max DD':>10} {'Trades':>8} {'Win Rate':>10}")
print("-"*90)

for key, res in sorted_results:
    print(f"{res['name']:<40} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Buy & Hold':<40} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8} {'-':>10}")
print("="*90)

# Best strategy
best = max(results.values(), key=lambda x: x['total_return'])
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])

print(f"\n🏆 BEST RETURN: {best['name']} at {best['total_return']:.1f}%")
print(f"📊 BEST SHARPE: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

# Comparison to buy & hold
print("\nVS BUY & HOLD:")
for key, res in sorted_results[:5]:  # Top 5
    diff = res['total_return'] - bh_return
    status = "✅ BEAT" if diff > 0 else "❌ LOST"
    print(f"  {res['name']:<40} {status} by {abs(diff):.1f}%")

## 7. Equity Curves

In [ ]:
# Plot top strategies
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=3, color='black', linestyle='--', alpha=0.7)

# Top 5 strategies
colors = ['green', 'blue', 'orange', 'red', 'purple']
for (key, res), color in zip(sorted_results[:5], colors):
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8, color=color)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('STH-MVRV Z-Score Exits: Top 5 Strategies', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Exit Timing Analysis

In [ ]:
# Analyze when Z-score exits fired
print("\n" + "="*90)
print("Z-SCORE EXIT TIMING ANALYSIS")
print("="*90)

# Test Z730 > 1.5 and Z730 > 2.0
for key in ['z730_1.5', 'z730_2.0']:
    if key not in exit_strategies:
        continue
    
    exits, name = exit_strategies[key]
    if exits.sum() == 0:
        print(f"\n{name}: No exit signals")
        continue
    
    print(f"\n{name}:")
    print("-" * 70)
    
    # Get exit dates
    exit_dates = df[exits].index
    
    # Show first 15 exits
    for exit_date in exit_dates[:15]:
        exit_price = df.loc[exit_date, 'price']
        sth_val = df.loc[exit_date, 'mvrv_sth']
        z_val = df.loc[exit_date, 'sth_z_730']
        
        # Forward returns
        try:
            future_30d = df[df.index > exit_date].iloc[30]['price'] if len(df[df.index > exit_date]) > 30 else None
            future_90d = df[df.index > exit_date].iloc[90]['price'] if len(df[df.index > exit_date]) > 90 else None
            
            ret_30d = (future_30d / exit_price - 1) * 100 if future_30d else None
            ret_90d = (future_90d / exit_price - 1) * 100 if future_90d else None
            
            print(f"  {exit_date.date()}: ${exit_price:,.0f} | STH-MVRV: {sth_val:.3f} | Z: {z_val:.2f}")
            if ret_30d:
                print(f"    30d: {ret_30d:+.1f}% | 90d: {ret_90d:+.1f}%")
        except:
            pass
    
    if len(exit_dates) > 15:
        print(f"  ... and {len(exit_dates) - 15} more signals")

print("\n" + "="*90)

## 9. Performance by Time Period

In [ ]:
# Test by period
periods = [
    ('2013-01-01', '2015-12-31', '2013-2015 (Early)'),
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY TIME PERIOD")
print("="*100)

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_entries = entries[(entries.index >= start) & (entries.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    # Test key strategies
    period_results = {}
    for key in ['never', 'z730_1.5', 'z730_2.0', 'z365_1.5', 'original']:
        if key not in exit_strategies:
            continue
        period_exits = exit_strategies[key][0][(exit_strategies[key][0].index >= start) & (exit_strategies[key][0].index <= end)]
        
        try:
            pf = vbt.Portfolio.from_signals(
                close=period_df['price'], entries=period_entries, exits=period_exits,
                fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
            )
            period_results[key] = pf.total_return() * 100
        except:
            period_results[key] = 0
    
    # Z-score stats
    period_z = period_df['sth_z_730'].dropna()
    overheated = (period_z > 2.0).sum()
    warming = ((period_z > 1.5) & (period_z <= 2.0)).sum()
    
    print(f"\n{label}:")
    print(f"  Z > 2.0 days: {overheated} ({overheated / len(period_z) * 100:.1f}%)")
    print(f"  Z > 1.5 days: {warming} ({warming / len(period_z) * 100:.1f}%)")
    print(f"  Buy & Hold: {bh:+.1f}%")
    
    for key, ret in sorted(period_results.items(), key=lambda x: x[1], reverse=True):
        name = exit_strategies[key][1]
        print(f"  {name}: {ret:+.1f}% ({ret - bh:+.1f}% vs B&H)")
    
    best_key = max(period_results, key=period_results.get)
    print(f"  🏆 Winner: {exit_strategies[best_key][1]}")

print("\n" + "="*100)

## 10. Final Verdict

In [ ]:
print("\n" + "="*90)
print("FINAL VERDICT: STH-MVRV Z-SCORE EXITS")
print("="*90)

never = results['never']
best = max(results.values(), key=lambda x: x['total_return'])
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])

# Get Z-score strategies
z730_1_5 = results.get('z730_1.5')
z730_2_0 = results.get('z730_2.0')
z365_1_5 = results.get('z365_1.5')

print(f"\n1. PERFORMANCE COMPARISON:")
print(f"   Buy & Hold:    {bh_return:.1f}%")
print(f"   Never Exit:    {never['total_return']:.1f}% ({never['total_return'] - bh_return:+.1f}%)")
if z730_1_5:
    print(f"   Z730 > 1.5:    {z730_1_5['total_return']:.1f}% ({z730_1_5['total_return'] - bh_return:+.1f}%)")
if z730_2_0:
    print(f"   Z730 > 2.0:    {z730_2_0['total_return']:.1f}% ({z730_2_0['total_return'] - bh_return:+.1f}%)")
if z365_1_5:
    print(f"   Z365 > 1.5:    {z365_1_5['total_return']:.1f}% ({z365_1_5['total_return'] - bh_return:+.1f}%)")
print(f"\n   🏆 Best Return: {best['name']} at {best['total_return']:.1f}%")
print(f"   📊 Best Sharpe: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

print(f"\n2. KEY INSIGHTS:")

if z730_1_5 and z730_2_0:
    improvement_1_5 = z730_1_5['total_return'] - never['total_return']
    improvement_2_0 = z730_2_0['total_return'] - never['total_return']
    
    print(f"   Z730 > 1.5 vs Never Exit: {improvement_1_5:+.1f}%")
    print(f"   Z730 > 2.0 vs Never Exit: {improvement_2_0:+.1f}%")
    
    if improvement_1_5 > 50 or improvement_2_0 > 50:
        print(f"   ✅ Z-score exits add MASSIVE value!")
    elif improvement_1_5 > 20 or improvement_2_0 > 20:
        print(f"   ✅ Z-score exits add significant value!")
    elif improvement_1_5 > 5 or improvement_2_0 > 5:
        print(f"   ✓ Z-score exits add modest value")
    else:
        print(f"   ❌ Z-score exits don't improve performance")

print(f"\n3. RISK-ADJUSTED METRICS:")
print(f"   Never Exit: Sharpe {never['sharpe']:.2f}, DD {never['max_dd']:.1f}%")
if z730_1_5:
    print(f"   Z730 > 1.5: Sharpe {z730_1_5['sharpe']:.2f}, DD {z730_1_5['max_dd']:.1f}%")
if z730_2_0:
    print(f"   Z730 > 2.0: Sharpe {z730_2_0['sharpe']:.2f}, DD {z730_2_0['max_dd']:.1f}%")

print(f"\n4. RECOMMENDATION:")

if best['name'] == 'Never Exit':
    print(f"   🎯 Optimal: Use Check's entries, NEVER EXIT")
    print(f"   📝 Z-score exits don't improve absolute returns")
    if best_sharpe['name'] != 'Never Exit':
        print(f"   💡 Consider {best_sharpe['name']} for better risk-adjusted returns")
elif 'Z730' in best['name'] or 'Z365' in best['name']:
    print(f"   🎯 Optimal: Use Check's entries + {best['name']}")
    print(f"   📝 Z-score exits successfully time local tops!")
    print(f"   ✅ This adapts to market regimes better than fixed thresholds")
else:
    print(f"   🎯 Optimal: {best['name']}")

# Final assessment
if best['total_return'] > bh_return:
    print(f"\n   🏆 SUCCESS: Beats buy-and-hold by {best['total_return'] - bh_return:.1f}%!")
else:
    gap = bh_return - best['total_return']
    print(f"\n   ⚠️  Still trails buy-and-hold by {gap:.1f}%")
    if best_sharpe['sharpe'] > 1.0:
        print(f"   But achieves Sharpe {best_sharpe['sharpe']:.2f} (better risk-adjusted)")

print("\n" + "="*90)

## Conclusion

**Z-Score normalization solves the regime problem:**

- Fixed thresholds (STH-MVRV > 1.7) don't adapt to changing market structure
- Z-scores measure "how extreme" relative to recent history
- What was overheated in 2017 might be normal in 2024

**Key findings:**
1. Does Z-score timing beat "Never Exit"?
2. Which lookback window works best (365d, 730d, expanding)?
3. Which Z-threshold is optimal (1.5, 2.0, 2.5)?
4. Can we finally beat buy-and-hold with regime-adaptive exits?

This is the most sophisticated exit strategy tested so far - it combines:
- Check's framework (STH-MVRV)
- Statistical normalization (Z-scores)
- Regime adaptation (rolling windows)
- Risk management (percentile-based exits)